In [1]:
from google.colab import files
files.upload()

Saving modelling_after_pp_V2.csv to modelling_after_pp_V2.csv


In [2]:
from huggingface_hub import notebook_login

notebook_login()

In [10]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("xlnet-base-cased")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

In [11]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

In [12]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [13]:
%pip install datasets
from datasets import *

ds = load_dataset("csv", data_files="/content/modelling_after_pp_V2.csv")

train_testvalid = ds['train'].train_test_split(test_size=0.2)
# Split the 10% test + valid in half test, half valid
test_valid = train_testvalid['test'].train_test_split(test_size=0.5)
# gather everyone if you want to have a single DatasetDict
ds = DatasetDict({
    'train': train_testvalid['train'],
    'test': test_valid['test'],
    'valid': test_valid['train']})

tokenized_imdb = ds.map(preprocess_function, batched=True)

Map:   0%|          | 0/295768 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Map:   0%|          | 0/36971 [00:00<?, ? examples/s]

Map:   0%|          | 0/36971 [00:00<?, ? examples/s]

In [14]:
%pip install evaluate
import evaluate

accuracy = evaluate.load("accuracy")

In [15]:
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [16]:
id2label ={
    0: 'sadness',
    1: 'joy',
    2: 'love',
    3: 'anger',
    4: 'fear',
    5: 'surprise'
}

label2id = {"sadness": 0, "joy": 1, "love": 2, "anger": 3, "fear": 4, "surprise": 5}

In [17]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    "xlnet-base-cased", num_labels=6, id2label=id2label, label2id=label2id
)

pytorch_model.bin:   0%|          | 0.00/467M [00:00<?, ?B/s]

Some weights of XLNetForSequenceClassification were not initialized from the model checkpoint at xlnet-base-cased and are newly initialized: ['logits_proj.bias', 'logits_proj.weight', 'sequence_summary.summary.bias', 'sequence_summary.summary.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
%pip uninstall -y accelerate
%pip uninstall -y transformers
%pip install -U accelerate
%pip install -U transformers
training_args = TrainingArguments(
    output_dir="my_awesome_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_imdb["train"],
    eval_dataset=tokenized_imdb["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Found existing installation: accelerate 0.30.1
Uninstalling accelerate-0.30.1:
  Successfully uninstalled accelerate-0.30.1
Found existing installation: transformers 4.41.1
Uninstalling transformers-4.41.1:
  Successfully uninstalled transformers-4.41.1
  Using cached accelerate-0.30.1-py3-none-any.whl (302 kB)


  Using cached transformers-4.41.1-py3-none-any.whl (9.1 MB)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.026600,0.026617,0.993968
2,0.013400,0.021242,0.995050


TrainOutput(global_step=36972, training_loss=0.042199645299848564, metrics={'train_runtime': 7904.2958, 'train_samples_per_second': 74.837, 'train_steps_per_second': 4.677, 'total_flos': 1.7191497093274656e+16, 'train_loss': 0.042199645299848564, 'epoch': 2.0})

In [19]:
from transformers import pipeline
text = ["When i think about school i feel depressed", "finally done with finals", "i feel loved", "i cant believe they did that", "so scared about today", "stunned by the exam today"]
classifier = pipeline("sentiment-analysis", model="/content/my_awesome_model")

for t in text:
  print(classifier(t))

[{'label': 'sadness', 'score': 0.9999709129333496}]
[{'label': 'joy', 'score': 0.9962097406387329}]
[{'label': 'love', 'score': 0.9999817609786987}]
[{'label': 'anger', 'score': 0.4730566740036011}]
[{'label': 'fear', 'score': 0.9998457431793213}]
[{'label': 'surprise', 'score': 0.9999352693557739}]


In [20]:
text = "so bad that it is good"

classifier(text)

[{'label': 'anger', 'score': 0.5869637131690979}]

In [21]:
text = "so bad that it is actually good"

classifier(text)

[{'label': 'anger', 'score': 0.6056511998176575}]

In [22]:
text = "expected it to be bad and still disappointed"

classifier(text)

[{'label': 'sadness', 'score': 0.5412829518318176}]

In [23]:
text = "expected it to be bad but was amazing"

classifier(text)

[{'label': 'joy', 'score': 0.6953315734863281}]

In [24]:
text = "Three years later, i still feel sad"

classifier(text)

[{'label': 'sadness', 'score': 0.9999731779098511}]

In [25]:
text = "Three years later, i feel good again"

classifier(text)

[{'label': 'joy', 'score': 0.9999833106994629}]